# Creating Groups with rules

In [218]:
import pandas as pd

In [219]:
user_features= pd.read_csv('user_features_data/user_features_cleaned.csv')

groups I want to create:
- high spenders
- maybe discount hunters? could be too small
- no bookers (gets also rid of all the NaNs in the data)
- seniors married/single?
- young & single?
- people without return flight?

In [220]:
#calculate total average revenue -> similar to average per day total?

In [221]:
user_features.describe().T

,count,mean,std,min,25%,50%,75%,max
user_id,5782.0,547670.236077,64035.394540,94883.000,519413.750000,542279.500000,576215.500000,844489.00000
Age,5782.0,43.887928,12.037172,19.000,37.000000,44.000000,51.000000,91.00000
days_in,5782.0,1270.738499,34.194997,1154.000,1255.000000,1272.000000,1283.000000,1619.00000
trip_session_ratio,5274.0,2.808584,1.613452,1.000,1.750000,2.333333,3.458333,10.00000
no_cancellations,5782.0,0.106192,0.318055,0.000,0.000000,0.000000,0.000000,2.00000
ave_page_clicks,5782.0,17.646602,8.612828,4.125,12.555556,15.625000,19.750000,109.12500
ave_discount,5782.0,0.072654,0.078744,0.000,0.000000,0.050000,0.125000,0.50000
hotel_median_nights,5782.0,4.057765,2.587651,1.000,2.500000,3.500000,5.000000,30.00000
hotel_tot_nights,5180.0,11.298842,7.852416,0.000,6.000000,10.000000,15.000000,63.00000
hotel_avg_nights,5782.0,4.458356,2.596641,1.000,3.000000,4.000000,5.000000,30.00000


## High spenders
no. 1 priority -> more money to make

In [222]:
user_features["ave_rev_per_day"].describe()

count    5782.000000
mean        2.862388
std         2.981440
min         0.000000
25%         1.029466
50%         2.192646
75%         3.843406
max        61.841390
Name: ave_rev_per_day, dtype: float64

In [223]:
outlier_limit = (user_features["ave_rev_per_day"].quantile(0.75) - user_features["ave_rev_per_day"].quantile(0.25))*1.5 + user_features["ave_rev_per_day"].quantile(0.75)

group_high_spenders = user_features[user_features["ave_rev_per_day"] > outlier_limit].copy() #for export
user_features["group_high_spenders"] = user_features["ave_rev_per_day"] > outlier_limit  #for overview in table
rest = user_features[~user_features["group_high_spenders"]] #no user should be in two groups
rest.drop('group_high_spenders', axis=1, inplace=True)

In [224]:
group_high_spenders

,user_id,gender,married,has_children,home_country,Age,days_in,trip_session_ratio,no_cancellations,ave_page_clicks,...,hotel_avg_rev,hotels_no_bookings,hotel_avg_cost_per_r_n,average_rooms,flight_no_bookings,flight_avg_seats,flight_avg_rev,flight_avg_cost_per_seat,ave_rev_per_day,flight_ave_days_stayed
18,299476,F,True,False,usa,41,1420,2.500000,1,39.000000,...,5731.00,2.0,601.5000,1.00,2.0,1.500000,2974.310000,1535.250000,12.261000,7.500
25,338086,F,False,True,usa,42,1389,2.250000,0,16.416667,...,376.50,4.0,152.2500,1.75,4.0,1.750000,2537.272500,849.735000,8.390994,2.250
27,347778,F,False,False,usa,51,1381,1.444444,0,22.923077,...,1448.50,8.0,210.5875,1.50,8.0,1.375000,1786.256375,760.931375,18.738632,2.875
36,359080,F,True,False,usa,44,1373,1.571429,0,18.727273,...,1842.25,4.0,317.7500,1.50,6.0,1.166667,1627.048333,998.490000,12.477269,5.400
39,373489,F,True,True,usa,39,1363,4.000000,0,14.250000,...,4747.50,2.0,220.0000,1.50,2.0,3.500000,12374.175000,3193.311250,25.123514,11.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5606,645168,F,True,False,usa,40,1225,2.666667,0,12.000000,...,413.40,2.0,174.9000,1.00,3.0,1.333333,3459.390667,1864.323667,9.146916,6.000
5642,654259,F,True,False,usa,48,1222,1.600000,0,16.125000,...,956.60,5.0,143.6000,1.40,5.0,1.400000,1055.016000,657.970000,8.230835,4.600
5661,659161,F,False,False,usa,41,1220,1.600000,0,15.250000,...,766.50,4.0,165.0000,1.75,5.0,1.600000,1778.416000,574.864000,9.801705,3.600
5670,663565,F,False,False,usa,43,1218,2.000000,0,13.000000,...,4146.75,4.0,307.5000,1.50,3.0,1.333333,640.340167,454.133500,15.195419,3.000


## no bookers
chose this for nr two priority, so that I don't have any nan values in the other groups for later analysis
Also, they booked nothing -> more work to get them to book than the others.

In [225]:
rest[["flight_no_bookings", "hotels_no_bookings"]].isna().value_counts()

flight_no_bookings  hotels_no_bookings
False               False                 4546
True                True                   508
                    False                  373
False               True                    94
Name: count, dtype: int64

- 4553 booked both - hotel & flight
- 508 booked nothing
- 383 booked only hotel
- 94 booked only flights

In [226]:
group_no_bookers = rest[rest["flight_no_bookings"].isna() & rest["hotels_no_bookings"].isna()].copy()
user_features['group_no_bookers'] = user_features["flight_no_bookings"].isna() & user_features["hotels_no_bookings"].isna()

In [227]:
group_only_hotel_bookers = rest[rest["flight_no_bookings"].isna() & ~rest['hotels_no_bookings'].isna()].copy()
user_features['group_only_hotel_bookers'] = user_features["flight_no_bookings"].isna()

In [228]:
group_only_flight_bookers = rest[rest["hotels_no_bookings"].isna() & ~rest["flight_no_bookings"].isna()].copy()
user_features['group_only_flight_bookers'] = user_features["hotels_no_bookings"].isna()

In [229]:
#again the rest. the groups above didn't interfere one another
exclude = group_no_bookers.index.tolist() + group_only_hotel_bookers.index.tolist() + group_only_flight_bookers.index.tolist()
rest = rest[~rest.index.isin(exclude)]

In [230]:
rest

,user_id,gender,married,has_children,home_country,Age,days_in,trip_session_ratio,no_cancellations,ave_page_clicks,...,hotel_avg_rev,hotels_no_bookings,hotel_avg_cost_per_r_n,average_rooms,flight_no_bookings,flight_avg_seats,flight_avg_rev,flight_avg_cost_per_seat,ave_rev_per_day,flight_ave_days_stayed
0,94883,F,True,False,usa,54,1619,3.666667,0,8.333333,...,115.000000,2.0,90.000000,1.500000,3.0,1.666667,3489.600000,1784.953333,6.608277,4.333333
1,101486,F,True,True,usa,53,1609,3.333333,1,26.153846,...,880.000000,3.0,163.666667,1.333333,2.0,1.000000,175.871250,175.871250,1.859380,3.500000
2,101961,F,True,False,usa,45,1609,1.714286,0,18.166667,...,502.857143,7.0,150.285714,1.000000,6.0,1.000000,320.705500,320.705500,3.383613,4.333333
3,106907,F,True,True,usa,47,1602,1.500000,1,26.285714,...,1092.600000,2.0,182.600000,1.000000,1.0,1.000000,165.510000,165.510000,1.467360,NaN
4,118043,F,False,True,usa,54,1588,2.200000,1,16.153846,...,1610.862500,4.0,218.850000,1.250000,3.0,2.000000,1708.853333,779.763333,7.285901,4.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5776,777366,F,False,False,usa,51,1178,5.000000,0,16.500000,...,628.000000,1.0,314.000000,1.000000,1.0,1.000000,108.730000,108.730000,0.625407,3.000000
5777,780167,F,True,True,usa,52,1177,2.500000,0,15.500000,...,220.000000,1.0,110.000000,2.000000,2.0,1.500000,798.591000,561.053000,1.543910,2.000000
5778,785186,F,True,True,usa,47,1175,4.000000,0,21.625000,...,237.500000,2.0,145.500000,1.000000,2.0,1.000000,176.675000,176.675000,0.704979,2.500000
5779,792549,F,False,False,usa,48,1172,2.000000,0,14.250000,...,180.000000,1.0,36.000000,1.000000,4.0,1.000000,259.792500,259.792500,1.040247,3.000000


In [231]:
user_features.head()

,user_id,gender,married,has_children,home_country,Age,days_in,trip_session_ratio,no_cancellations,ave_page_clicks,...,flight_no_bookings,flight_avg_seats,flight_avg_rev,flight_avg_cost_per_seat,ave_rev_per_day,flight_ave_days_stayed,group_high_spenders,group_no_bookers,group_only_hotel_bookers,group_only_flight_bookers
0,94883,F,True,False,usa,54,1619,3.666667,0,8.333333,...,3.0,1.666667,3489.600000,1784.953333,6.608277,4.333333,False,False,False,False
1,101486,F,True,True,usa,53,1609,3.333333,1,26.153846,...,2.0,1.000000,175.871250,175.871250,1.859380,3.500000,False,False,False,False
2,101961,F,True,False,usa,45,1609,1.714286,0,18.166667,...,6.0,1.000000,320.705500,320.705500,3.383613,4.333333,False,False,False,False
3,106907,F,True,True,usa,47,1602,1.500000,1,26.285714,...,1.0,1.000000,165.510000,165.510000,1.467360,NaN,False,False,False,False
4,118043,F,False,True,usa,54,1588,2.200000,1,16.153846,...,3.0,2.000000,1708.853333,779.763333,7.285901,4.666667,False,False,False,False


## no return flight (ever)

In [244]:
group_no_return = rest[rest['flight_ave_days_stayed'].isna()].copy()
user_features['group_no_return'] = user_features["flight_ave_days_stayed"].isna()

In [249]:
rest = rest[~rest.index.isin(group_no_return.index)]
group_no_return.head()

,user_id,gender,married,has_children,home_country,Age,days_in,trip_session_ratio,no_cancellations,ave_page_clicks,...,hotel_avg_rev,hotels_no_bookings,hotel_avg_cost_per_r_n,average_rooms,flight_no_bookings,flight_avg_seats,flight_avg_rev,flight_avg_cost_per_seat,ave_rev_per_day,flight_ave_days_stayed
3,106907,F,True,True,usa,47,1602,1.5,1,26.285714,...,1092.6,2.0,182.600,1.0,1.0,1.0,165.51,165.51,1.467360,NaN
209,487352,F,False,True,usa,20,1300,3.5,0,9.666667,...,538.2,2.0,87.525,1.5,1.0,1.0,295.39,295.39,1.055223,NaN
272,499669,F,True,True,usa,34,1293,4.0,0,7.818182,...,858.0,1.0,286.000,1.0,1.0,1.0,168.98,168.98,0.794261,NaN
369,510558,M,False,True,usa,25,1288,6.0,1,16.222222,...,228.0,1.0,114.000,1.0,1.0,1.0,359.89,359.89,0.456436,NaN
487,513289,F,False,False,canada,37,1286,4.0,0,15.500000,...,1311.0,1.0,437.000,1.0,1.0,1.0,189.31,189.31,1.166649,NaN


# Export groups to CSV

In [247]:
group_high_spenders.to_csv('groups_data/high_spenders.csv',index=False)
group_no_bookers.to_csv('groups_data/no_bookers.csv',index=False)
group_only_flight_bookers.to_csv('groups_data/only_flight_bookers.csv',index=False)
group_only_hotel_bookers.to_csv('groups_data/only_hotel_bookers.csv',index=False)
group_no_return.to_csv('groups_data/no_return.csv',index=False)
#rest.to_csv('groups_data/rest.csv',index=False)

In [234]:
group_no_bookers.head()

,user_id,gender,married,has_children,home_country,Age,days_in,trip_session_ratio,no_cancellations,ave_page_clicks,...,hotel_avg_rev,hotels_no_bookings,hotel_avg_cost_per_r_n,average_rooms,flight_no_bookings,flight_avg_seats,flight_avg_rev,flight_avg_cost_per_seat,ave_rev_per_day,flight_ave_days_stayed
6,167852,F,False,False,usa,20,1534,NaN,0,10.692308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
11,217114,O,True,True,usa,22,1489,NaN,1,26.454545,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
12,228195,F,False,True,usa,29,1480,NaN,1,21.909091,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
19,306165,F,False,False,usa,23,1414,NaN,0,12.545455,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
53,382533,F,False,False,usa,20,1357,NaN,0,7.181818,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN


In [235]:
group_only_flight_bookers.head()

,user_id,gender,married,has_children,home_country,Age,days_in,trip_session_ratio,no_cancellations,ave_page_clicks,...,hotel_avg_rev,hotels_no_bookings,hotel_avg_cost_per_r_n,average_rooms,flight_no_bookings,flight_avg_seats,flight_avg_rev,flight_avg_cost_per_seat,ave_rev_per_day,flight_ave_days_stayed
120,452532,F,False,False,usa,35,1317,1.0,1,34.000,...,NaN,NaN,NaN,NaN,2.0,1.0,320.58375,320.58375,0.486839,2.0
405,511568,F,False,False,usa,23,1287,8.0,0,13.375,...,NaN,NaN,NaN,NaN,1.0,1.0,1229.22000,1229.22000,0.955105,15.0
461,512704,F,True,True,usa,32,1286,7.0,0,14.500,...,NaN,NaN,NaN,NaN,1.0,1.0,380.99750,380.99750,0.296266,1.0
589,515818,M,False,False,usa,69,1285,5.0,0,17.500,...,NaN,NaN,NaN,NaN,1.0,2.0,2056.22000,1028.11000,1.600171,5.0
610,516217,M,True,False,canada,47,1285,1.0,0,23.250,...,NaN,NaN,NaN,NaN,1.0,1.0,509.57000,509.57000,0.396553,4.0


In [236]:
group_only_hotel_bookers.head()

,user_id,gender,married,has_children,home_country,Age,days_in,trip_session_ratio,no_cancellations,ave_page_clicks,...,hotel_avg_rev,hotels_no_bookings,hotel_avg_cost_per_r_n,average_rooms,flight_no_bookings,flight_avg_seats,flight_avg_rev,flight_avg_cost_per_seat,ave_rev_per_day,flight_ave_days_stayed
26,340166,M,True,True,usa,53,1387,7.0,0,9.181818,...,474.0,1.0,237.0,1.0,NaN,NaN,NaN,NaN,0.341745,NaN
55,386469,F,False,False,usa,19,1354,1.0,0,12.200000,...,4970.0,1.0,497.0,1.0,NaN,NaN,NaN,NaN,3.670606,NaN
84,429345,M,True,True,usa,68,1330,8.0,0,14.300000,...,714.0,1.0,119.0,1.0,NaN,NaN,NaN,NaN,0.536842,NaN
129,458725,F,False,False,usa,19,1314,2.0,0,10.400000,...,266.0,1.0,66.5,1.0,NaN,NaN,NaN,NaN,0.202435,NaN
161,471946,F,True,True,usa,47,1307,7.0,0,11.300000,...,282.0,1.0,94.0,1.0,NaN,NaN,NaN,NaN,0.215761,NaN


# Clustering with kmeans

## Preprocessing

In [250]:
df_preprocessed = rest.drop(["user_id"], axis=1).copy()

In [251]:
df_preprocessed.info()

<class 'pandas.DataFrame'>
Index: 4489 entries, 0 to 5780
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   gender                    4489 non-null   str    
 1   married                   4489 non-null   bool   
 2   has_children              4489 non-null   bool   
 3   home_country              4489 non-null   str    
 4   Age                       4489 non-null   int64  
 5   days_in                   4489 non-null   int64  
 6   trip_session_ratio        4489 non-null   float64
 7   no_cancellations          4489 non-null   int64  
 8   ave_page_clicks           4489 non-null   float64
 9   ave_discount              4489 non-null   float64
 10  hotel_median_nights       4489 non-null   float64
 11  hotel_tot_nights          4489 non-null   float64
 12  hotel_avg_nights          4489 non-null   float64
 13  hotel_avg_rev             4489 non-null   float64
 14  hotels_no_bookings      

In [239]:
# Show all columns
#pd.set_option('display.max_columns', None)

# Show all rows
#pd.set_option('display.max_rows', None)

#pd.reset_option('display.max_columns')
#pd.reset_option('display.max_rows')

## Encoding
Gender, home-country